# Example 05C: Compare Ensemble Outputs

This notebook compares completed sibling simulations from notebook 05B using the outlet flow-component files. The same workflow can compare one base model with many realizations, or several base models by loading multiple simulation manifests and assigning each one a `base_model` label.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "sewertris").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXAMPLES_DIR = PROJECT_ROOT / "examples"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import sewertris as sp

ENSEMBLE_ROOT = EXAMPLES_DIR / "output_example_5_ensembles"
SIMULATION_MANIFEST = ENSEMBLE_ROOT / "simulation_manifest.csv"
FIGURE_PATH = ENSEMBLE_ROOT / "ensemble_outlet_flow_comparison.png"

print("Simulation manifest:", SIMULATION_MANIFEST)

## Load Simulation Manifests

Add more entries to `MANIFEST_SOURCES` when comparing ensembles produced from different base models. Each manifest should contain the columns written by notebook 05B, including `ensemble`, `realization`, `project_file`, and `flows_path`.

In [ ]:
MANIFEST_SOURCES = [
    {
        "base_model": "Stillwater V2 base",
        "path": SIMULATION_MANIFEST,
    },
    # Example for a second base model:
    # {
    #     "base_model": "Stillwater alternate base",
    #     "path": EXAMPLES_DIR / "output_example_5_ensembles_alt" / "simulation_manifest.csv",
    # },
]

manifests = []
for source in MANIFEST_SOURCES:
    path = Path(source["path"])
    if not path.exists():
        print(f"Missing manifest: {path}")
        continue
    manifest = pd.read_csv(path)
    manifest["base_model"] = source["base_model"]
    manifest["manifest_path"] = str(path)
    manifests.append(manifest)

if manifests:
    simulation_manifest = pd.concat(manifests, ignore_index=True)
    simulation_manifest["flows_exists"] = simulation_manifest["flows_path"].map(lambda p: Path(p).exists())
    display(simulation_manifest)
else:
    simulation_manifest = pd.DataFrame()
    print("No simulation manifests were loaded. Run notebook 05B first.")

## Ensemble Dashboard

The top four panels show outlet hydrographs for total flow, GWI, BWF/DWF, and RDII. Thin lines are individual simulations, and the heavier line is the group median with a quantile band. The bottom section uses six panels: BWF/DWF, GWI, and RDII volumes on one row, then BWF/DWF, GWI, and RDII peak flows on the next row, each with its own y-axis scale.

In [ ]:
START = None
END = None

if "flows_exists" in simulation_manifest.columns:
    available_manifest = simulation_manifest.loc[simulation_manifest["flows_exists"]].copy()
else:
    available_manifest = pd.DataFrame()

if available_manifest.empty:
    print("No completed flow-component files were found. Run notebook 05B first.")
else:
    fig, ensemble_summary = sp.plot_ensemble_results(
        available_manifest,
        group_cols=("base_model", "ensemble"),
        start=START,
        end=END,
        title="Stillwater Ensemble Outlet Flow Components",
        savepath=FIGURE_PATH,
        return_summary=True,
    )
    print("Saved figure:", FIGURE_PATH)
    display(ensemble_summary.head())

## Summary Tables

In [ ]:
if "ensemble_summary" in globals() and not ensemble_summary.empty:
    volume_table = ensemble_summary.pivot_table(
        index=["base_model", "ensemble", "realization"],
        columns="component",
        values="volume",
    )
    peak_table = ensemble_summary.pivot_table(
        index=["base_model", "ensemble", "realization"],
        columns="component",
        values="peak_flow",
    )

    print("Component volumes")
    display(volume_table)
    print("Component peak flows")
    display(peak_table)
else:
    print("Run the dashboard cell first.")